# Sprint Block H — Base-Paper Reproduction + Adjective Extension

**Goal of this notebook:** beat Yousaf, Schwartz & Wolter (2023) at *their own task, on their own
dataset, with their own metric* — by adding the adjective layer their paper explicitly discards.

### Why this notebook exists

The earlier Blocks E+F+G solved **geocell prediction** (*"which 55–880 km patch of Earth is this?"*).
That is **not** the thesis task and cannot be compared to the base paper. The proposal — and the base
paper — are about **entity typing**: *"what kind of place is this?"* → a ranked list of OSM tags
(`cafe`, `bakery`, `restaurant`, …). A paraphrased place ("a nice spot for ice-cream") has no name,
so you need its **type** before you can query OSM for it at all.

### The gap we exploit

> Yousaf et al. (2023), Algorithm 1, footnote 6:
> *"Currently, we ignore attribute adjectives, determiners etc."*
>
> …and from their Conclusions:
> *"Consideration of adjectives or verbs might further improve results… identifying a method to
> remove abstract nouns that are not informative for the place category (e.g., place, spot) may help."*

Our contribution is literally their stated future work.

### Their published results — the numbers to beat

| Model | Hits@1 | Hits@5 | Hits@10 | Hits@20 | nDCG@20 |
|---|---|---|---|---|---|
| GloVe | 0.0189 | 0.1792 | 0.3208 | 0.4623 | 0.1913 |
| W2V | 0.1604 | 0.4717 | 0.6509 | 0.7075 | 0.3968 |
| **W2V Clustered** | **0.2264** | **0.5283** | 0.6509 | 0.7075 | **0.4442** |
| BERT (DeBERTa) | 0.1038 | 0.3491 | 0.4245 | 0.5377 | 0.2911 |

### Task plan for this notebook

| Step | What it does |
|---|---|
| **H1** | Build the 106-row evaluation set (this section) |
| **H2** | GoogleNews Word2Vec + their Algorithm 2 (noun→tag similarity) & Algorithm 3 (clustering) |
| **H3** | **Reproduce the table above** — the credibility gate |
| **H4** | Fix `adjective_extractor.py`, run it on the 106 sentences |
| **H5** | Adjective extension, ablated → the RQ1 headline result |

---

# H1 — Build the evaluation set

We join **three** sources into one 106-row table. Getting this join right is load-bearing: every
number in H3 and H5 is computed against it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob
ROOT = '/content/drive/MyDrive/geoproject'
os.environ['HF_HOME'] = f'{ROOT}/models_cache/hf'   # HF models cached on Drive

# Block H writes here. `eval/` holds the reproduction dataset, `results_h/` the
# metrics -- kept separate from `results/` so the Block E/F/G geocell artifacts
# are preserved rather than overwritten.
for d in ('models', 'results', 'results_h', 'eval'):
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

print('ROOT =', ROOT)
print('geoproject/ contains:', sorted(os.listdir(ROOT)))


## H1.1 — The three data sources

| Source | Where from | What it gives us |
|---|---|---|
| `full_input_tokens.txt` | authors' public repo | 212 lines = **106 alternating pairs**: a line of context nouns, then the gold OSM tag. This is the authors' *own Algorithm 1 output* — they hand us their extracted nouns, so the noun-only baseline needs no re-extraction. **These gold tags are canonical.** |
| `osm_tags_complete.txt` | authors' public repo | 243 candidate OSM tags — the ranking search space |
| `testdatagisnew` | supplied by Dr. Yousaf | the **106 full sentences**, each with the gold tag, venue name, and ambiguity notes in parentheses. This is the piece the repo only ships as a `.odt`, and it is what makes the adjective extension possible — the repo's noun lists have already thrown the adjectives away. |

The repo is CC0 and public, so the first two are fetched live (reproducible). `testdatagisnew` is not
public, so it must be uploaded to `geoproject/eval/` on Drive once.

> **Note on the division of labour:** the authors' pre-extracted nouns give us a *faithful* noun-only
> baseline (H3) that cannot be accused of a weakened reproduction. The full sentences give us the
> adjectives they discarded (H4/H5). We need both.


In [ ]:
import urllib.request, textwrap, glob, re, os

EVAL = f'{ROOT}/eval'
REPO = 'https://raw.githubusercontent.com/dwolter/place-descriptions/main/paraphrased/'

# Cache the public files on Drive so a re-run is offline and the exact bytes used
# for the thesis numbers are frozen alongside the results.
for fn in ('full_input_tokens.txt', 'osm_tags_complete.txt'):
    dest = f'{EVAL}/{fn}'
    if not os.path.exists(dest):
        urllib.request.urlretrieve(REPO + fn, dest)
        print(f'downloaded  {fn}')
    else:
        print(f'cached      {fn}')

# The sentence file is not public and has been passed around under several names
# (testdatagisnew / testdata-gis-new / testdata_gis_new.txt), so match on a
# normalized form rather than one exact spelling.
def _key(p):
    return re.sub(r'[^a-z0-9]', '', os.path.basename(p).lower())

cands = [p for p in glob.glob(f'{EVAL}/*') if _key(p).startswith('testdatagisnew')]
if not cands:
    raise FileNotFoundError(textwrap.dedent(f"""
        No sentence file found in {EVAL}

        Expected the file Dr. Yousaf supplied: 106 numbered sentences with gold tags
        in parentheses, starting "1) This cake shop is one of the most photographed...".
        Any of these names work: testdatagisnew, testdata-gis-new, testdata_gis_new.txt
        Found instead: {[os.path.basename(p) for p in glob.glob(f'{EVAL}/*')]}"""))
SENT_FILE = max(cands, key=os.path.getsize)      # largest if several variants
print(f'sentences   {os.path.basename(SENT_FILE)}')

for p in (f'{EVAL}/full_input_tokens.txt', f'{EVAL}/osm_tags_complete.txt', SENT_FILE):
    n = sum(1 for _ in open(p, encoding='utf-8'))
    print(f'  {n:>5} lines  {os.path.getsize(p):>7,} bytes  {os.path.basename(p)}')


## H1.2 — Parse the gold pairs (canonical labels)

`full_input_tokens.txt` is strictly alternating, with no row markers:

```
tea, cupcakes                              <- line 1  : context nouns for row 1
cafe                                       <- line 2  : gold tag for row 1
crab linguine, honey, yogurt, place        <- line 3  : context nouns for row 2
restaurant                                 <- line 4  : gold tag for row 2
```

So line `2i-1` holds the nouns and line `2i` the gold tag for row `i`. 212 lines → 106 rows.

**Why these gold tags are the canonical ones:** the sentence file (`testdatagisnew`) sometimes lists
several plausible tags for one sentence (`bars/pubs`, `park and attraction`). `full_input_tokens.txt`
commits to exactly one per row — that single choice is what the authors scored their Table 4 against,
so it is what we must score against too. Alternatives from the sentence file are kept separately as
`alt_tags` for an optional lenient-match metric, never as the primary gold.


In [ ]:
lines = open(f'{EVAL}/full_input_tokens.txt', encoding='utf-8').read().split('\n')
lines = [l.rstrip() for l in lines]
while lines and not lines[-1].strip():      # trailing blank line in the published file
    lines.pop()

assert len(lines) % 2 == 0, f'file must alternate nouns/tag, got {len(lines)} lines'

gold = [{'row': i // 2 + 1,
         'context_nouns': [n.strip() for n in lines[i].split(',') if n.strip()],
         'gold_tag_raw': lines[i + 1].strip()}
        for i in range(0, len(lines), 2)]

print(f'{len(lines)} lines -> {len(gold)} rows\n')
for g in gold[:3]:
    print(f"  row {g['row']}: {g['context_nouns']}  ->  {g['gold_tag_raw']}")

_n = [len(g['context_nouns']) for g in gold]
print(f'\ncontext nouns per row: min {min(_n)}, mean {sum(_n)/len(_n):.2f}, max {max(_n)}')


## H1.3 — Parse the full sentences

`testdatagisnew` is hand-written free text. Each row looks like:

```
3)This parsian place on Elizabeth Street makes some of the best bread in the city.(Poilane) (shop bakery)
   \________________________ the sentence _________________________________/  \_ venue _/ \__ tag __/
```

but the annotation block varies a lot in practice:

| Shape seen | Example |
|---|---|
| key + value | `(amenity cafe)` |
| tag + location gloss | `(zoo in uk)` |
| venue + copula + tag | `(sketch is restaurant)` |
| several tags | `(bars/pubs)`, `(fastfood and bakery)` |
| pure ambiguity note | `(also a bar in canada)` |
| unparenthesised trailing prose | `(pub) also a building` |

**Splitting rule.** Annotations always come *after* the description, so we cut at the **last sentence
terminator that precedes the first `(`**. This keeps genuinely multi-sentence descriptions intact
(row 24 is two sentences; row 28 is three). Three rows have annotations embedded mid-sentence and have
no such terminator — those fall back to stripping the parenthetical groups in place and are **flagged**
for manual review rather than silently mangled.

We do **not** take the gold tag from this file — that comes from `full_input_tokens.txt` (H1.2). We only
need the sentence text here. The annotations are parsed anyway, purely as a *cross-check* that row `i`
of this file really is row `i` of the token file.


In [ ]:
import re

raw = open(SENT_FILE, encoding='utf-8').read()
raw = re.sub(r'#{4,}', '', raw)                 # the "#####" separator between blog batches

# Split on a leading "<n>)" at the start of a line -> {row_number: body}
chunks = re.split(r'(?m)^\s*(\d{1,3})\s*\)', raw)[1:]
sent_rows = {}
for n, body in zip(chunks[0::2], chunks[1::2]):
    body = ' '.join(body.split()).strip(' #')   # collapse newlines/whitespace
    if body:
        sent_rows[int(n)] = body

PAREN = re.compile(r'\(([^()]*)\)')

def split_sentence(body):
    """-> (sentence, [annotation strings], flagged_for_review)

    Cut at the last sentence terminator before the first '(' so multi-sentence
    descriptions survive intact. No terminator => annotations are embedded
    mid-sentence; strip the paren groups in place and flag the row.
    """
    first_paren = body.find('(')
    if first_paren == -1:
        return body.strip(), [], False
    head = body[:first_paren]
    terminators = list(re.finditer(r'[.!?]', head))
    if terminators:
        cut = terminators[-1].end()
        return body[:cut].strip(), PAREN.findall(body[cut:]), False
    sentence = ' '.join(PAREN.sub(' ', body).split()).strip()
    return sentence, PAREN.findall(body), True

print(f'{len(sent_rows)} numbered rows (min {min(sent_rows)}, max {max(sent_rows)})')
print('\nsplit examples:')
for n in (3, 24, 64):
    s, a, f = split_sentence(sent_rows[n])
    print(f'\n  [{n}]{"  <FLAGGED>" if f else ""}\n     sentence : {s[:110]}\n     annots   : {a}')


## H1.4 — Normalize the candidate tag lexicon

`osm_tags_complete.txt` is the search space: every ranking is a permutation of these tags. It ships
dirty, with the *same tag present several times* in different surface forms:

```
brewery / breweries      church / churches / cathedrals
street / Street / streets      town / Town      Theatre / theatre      University / university
```

Duplicates are actively harmful for ranking: if `brewery` and `breweries` are both candidates, they
split the similarity mass and can push the gold tag down a rank purely as an artefact.

**Two-stage cleanup:**
1. **Case / whitespace** — lowercase, collapse spaces and underscores to `-` (`fast food` → `fast-food`).
2. **Plurals** — collapse `X` → singular *only when the singular is itself already in the lexicon*.
   This is deliberately conservative: it removes real duplicates without inventing tags the authors
   never listed (e.g. `bus-stop` keeps its hyphen and is left alone).

The critical post-condition, asserted in H1.6: **every gold tag must survive into the final lexicon.**
A gold tag missing from the candidate list would be unrankable and would silently cap every metric.


In [ ]:
import re, unicodedata

def norm_tag(t):
    """Surface-form normalization: NFKD, lowercase, spaces/underscores -> hyphen."""
    t = unicodedata.normalize('NFKD', t).strip().lower()
    return re.sub(r'[\s_]+', '-', t)

lex_raw = [l.strip() for l in
           open(f'{EVAL}/osm_tags_complete.txt', encoding='utf-8').read().split('\n') if l.strip()]
lex_norm = {norm_tag(t) for t in lex_raw}

# Plural -> singular, only where the singular is already a listed tag.
singular = {}
for t in sorted(lex_norm):
    for suf, rep in (('ies', 'y'), ('ches', 'ch'), ('shes', 'sh'), ('s', '')):
        if t.endswith(suf):
            cand = t[:-len(suf)] + rep
            if cand in lex_norm and cand != t:
                singular[t] = cand
                break

lexicon = sorted(lex_norm - set(singular))

print(f'{len(lex_raw)} raw -> {len(lex_norm)} case/space-normalized -> {len(lexicon)} final')
print(f'\ncollapsed {len(singular)} plural duplicates:')
for k, v in sorted(singular.items()):
    print(f'    {k:<14} -> {v}')


## H1.5 — Join the sources, and parse annotations as a cross-check

Now we join row `i` of the token file to row `i` of the sentence file. The join is positional, so it
needs verifying — a silent off-by-one would corrupt every downstream number while still *looking* fine.

**The cross-check:** parse the tags out of each sentence's annotation block and ask whether they agree
with the canonical gold tag from `full_input_tokens.txt`. High agreement ⇒ the positional join is
sound. This is why we bother parsing annotations we otherwise don't need.

The parser has to reduce all the shapes from H1.3 down to a lexicon entry:

| Input | Handling | Result |
|---|---|---|
| `amenity cafe` | strip OSM keys anywhere in the string | `cafe` |
| `zoo in uk` | strip the `in <place>` location gloss | `zoo` |
| `sketch is restaurant` | strip everything up to the copula | `restaurant` |
| `correct type restaurant` | strip the editorial prefix | `restaurant` |
| `bars/pubs` | split on `/`, `,`, `and`, `or`; depluralize | `bar`, `pub` |
| `fastfood`, `restauants` | typo/spelling alias table | `fast-food`, `restaurant` |
| `also a bar in canada` | dropped — it is an ambiguity note, not a label | — |

Anything that does not land on a lexicon entry after all that is discarded rather than guessed.


In [ ]:
KEYS = ('amenity', 'shop', 'craft', 'leisure', 'natural', 'tourism', 'historic',
        'building', 'highway', 'place', 'landuse', 'railway')
DROP        = re.compile(r'^\s*(also\b|there are\b)', re.I)   # ambiguity note, not a label
STRIP_PRE   = re.compile(r'^\s*(?:correct\s+type|correct)\s+', re.I)
STRIP_COPULA= re.compile(r'^.*?\bis\s+(?:a\s+)?', re.I)       # 'sketch is restaurant'
STRIP_LOC   = re.compile(r'\s+in\s+\w.*$', re.I)              # 'zoo in uk' -> 'zoo'
KEY_ANY     = re.compile(r'\b(?:%s)\b' % '|'.join(KEYS), re.I)
ALIAS = {'fastfood': 'fast-food', 'fast food': 'fast-food', 'restauants': 'restaurant',
         'restaurents': 'restaurant', 'buidling': 'building', 'pedistrian': 'pedestrian'}

def depluralize(t):
    if t in lex_norm:
        return singular.get(t, t)
    for suf, rep in (('ies', 'y'), ('ches', 'ch'), ('shes', 'sh'), ('s', '')):
        if t.endswith(suf):
            cand = t[:-len(suf)] + rep
            if cand in lex_norm:
                return singular.get(cand, cand)
    return t

def canon(t):
    """Any surface form -> a lexicon entry, or '' if it isn't one."""
    t = ALIAS.get(t.strip().lower(), t)
    t = depluralize(ALIAS.get(norm_tag(t), norm_tag(t)))
    return t if t in lexicon else ''

def annot_tags(annots):
    out = []
    for a in annots:
        a = a.strip()
        if not a or DROP.match(a):
            continue
        a = STRIP_PRE.sub('', a)
        if re.search(r'\bis\b', a):
            a = STRIP_COPULA.sub('', a)
        a = KEY_ANY.sub(' ', a)
        for part in re.split(r'\s*(?:/|,|\bor\b|\band\b)\s*', a):
            t = canon(STRIP_LOC.sub('', part))
            if t:
                out.append(t)
    return sorted(set(out))

rows = []
for g in gold:
    sentence, annots, flagged = split_sentence(sent_rows[g['row']])
    gt = canon(g['gold_tag_raw']) or depluralize(norm_tag(g['gold_tag_raw']))
    rows.append({'row': g['row'], 'sentence': sentence,
                 'context_nouns': g['context_nouns'], 'gold_tag': gt,
                 'alt_tags': [t for t in annot_tags(annots) if t != gt],
                 'annotations': annots, 'flagged': flagged})

print(f'joined {len(rows)} rows\n')
for r in rows[:2]:
    print(f"  [{r['row']}] gold={r['gold_tag']}  nouns={r['context_nouns']}")
    print(f"        {r['sentence'][:100]}")


## H1.6 — Validation, class balance, and the baselines the original paper omits

Four assertions must hold before any H2/H3 number is trustworthy:

1. **106 rows**, one per gold pair.
2. **Every gold tag is in the lexicon** — otherwise it is unrankable and caps the metric silently.
3. **Every row has a sentence** — the positional join found a partner for each.
4. **High annotation agreement** — independent evidence the positional join is correctly phased.

### And a control the base paper does not report

The 106 rows are severely imbalanced (26 classes, over half the rows in the top 4). That makes
**Hits@1 easy to misread**, so we compute two trivial baselines and carry them into every results table:

- **Majority class** — always answer the single most common tag. Uses only a domain prior
  ("restaurants dominate food blogs"), so it is a fair, honest control.
- **Frequency prior** — rank all tags by their frequency in this set. This *peeks at the test
  distribution* (there is no train split — the 106 rows are the whole set), so it is an **upper bound
  on what priors alone can achieve**, not a legitimate competing system. Reported as a ceiling only.

Adding a missing control is a contribution, not an attack: it does not diminish the authors' method
(which does semantic reasoning, not prior-fitting), it tells us **which metrics on this dataset carry
real information** — and that shapes how we should report our own results in H5.


In [ ]:
from collections import Counter

gold_tags = {r['gold_tag'] for r in rows}
oov       = sorted(t for t in gold_tags if t not in lexicon)
no_sent   = [r['row'] for r in rows if not r['sentence']]
flagged   = [r['row'] for r in rows if r['flagged']]
agree     = sum(1 for r in rows if r['gold_tag'] in annot_tags(r['annotations']))
contra    = [(r['row'], r['gold_tag'], annot_tags(r['annotations'])) for r in rows
             if annot_tags(r['annotations']) and r['gold_tag'] not in annot_tags(r['annotations'])]

assert len(rows) == 106,  f'expected 106 rows, got {len(rows)}'
assert not oov,           f'gold tags missing from lexicon (unrankable): {oov}'
assert not no_sent,       f'rows with no sentence: {no_sent}'

print(f'rows                          : {len(rows)}')
print(f'distinct gold tags            : {len(gold_tags)}')
print(f'gold tags outside lexicon     : none  (assertion passed)')
print(f'rows missing a sentence       : none  (assertion passed)')
print(f'annotation corroborates gold  : {agree}/{len(rows)}   <- join phasing check')
print(f'flagged (annots mid-sentence) : {flagged}')
print(f'annotation contradicts gold   : {[c[0] for c in contra]}')
for row_n, g, a in contra:
    print(f'    row {row_n}: canonical={g!r}  annotation says={a}')

dist = Counter(r['gold_tag'] for r in rows)
print(f'\nclass distribution ({len(dist)} classes, '
      f'{sum(1 for c in dist.values() if c == 1)} singletons):')
for t, c in dist.most_common():
    print(f'   {c:>3}  {"#" * c}  {t}')

# ---- baselines the base paper does not report -------------------------------
N = len(rows)
top_tag, top_n = dist.most_common(1)[0]
by_freq = [t for t, _ in dist.most_common()]
print(f'\n{"BASELINES":-^66}')
print(f'Majority class (always "{top_tag}")   Hits@1 = {top_n}/{N} = {top_n/N:.4f}')
print(f'   Yousaf 2023 best published        Hits@1 =        {0.2264:.4f}   <-- LOWER')
for k in (1, 5, 10, 20):
    hit = sum(dist[t] for t in by_freq[:k])
    print(f'Frequency prior (test-peeking)      Hits@{k:<2} = {hit}/{N} = {hit/N:.4f}')
print(f'   Yousaf 2023 best published        Hits@5 =        {0.5283:.4f}')

json.dump({'rows': rows, 'lexicon': lexicon, 'plural_map': singular,
           'baselines': {'majority_tag': top_tag, 'majority_hits1': top_n / N,
                         'freq_prior_order': by_freq}},
          open(f'{EVAL}/evalset_106.json', 'w'), indent=1, ensure_ascii=False)
print(f'\nwrote {EVAL}/evalset_106.json')


In [ ]:
import textwrap
# Manual spot-check: the join is positional, so read these and confirm the sentence
# really is describing a place of type `gold`, using those `nouns`.
for i in (3, 5, 11, 29, 59, 64, 84, 102):
    r = rows[i - 1]
    tail = (f"   alt={r['alt_tags']}" if r['alt_tags'] else '') + ('   <FLAGGED>' if r['flagged'] else '')
    print(f"[{r['row']:>3}] GOLD: {r['gold_tag']}{tail}")
    print(f"      nouns : {', '.join(r['context_nouns'])}")
    print(textwrap.fill(r['sentence'], 100, initial_indent='      sent  : ',
                        subsequent_indent='              '))
    print()


---

# H2 — Word2Vec + the authors' Algorithm 2 and Algorithm 3

We now build the *base paper's own ranker*, faithfully. Nothing of ours goes in yet — H2 and H3
exist purely to prove we can reproduce their published numbers. Only once that passes does the
adjective extension (H4/H5) mean anything.

### Which embedding, and why it matters

The paper tried GloVe, Word2Vec and BERT (DeBERTa). Their results split sharply:

| Model | nDCG@20 |
|---|---|
| GloVe | 0.1913 |
| **Word2Vec Clustered** | **0.4442** ← their best |
| BERT (DeBERTa) | 0.2911 |

**Word2Vec is their best by a wide margin**, so it is the arm we must reproduce and the bar our
method has to clear. (An earlier attempt in the old Block G notebook used spaCy `en_core_web_md`
— that is *GloVe*, their **worst** embedding at nDCG 0.19. Comparing against that would have been
a straw man.)

Specifically: `GoogleNews-vectors-negative300` — 3M words, 300 dimensions, trained on ~100B words,
which is what the paper's reference [18] (Mikolov et al. 2013) points to.

### Caching strategy

The full model is 3.6 GB, but this task only needs vectors for **227 OSM tags + ~420 distinct
context nouns**. So we download the full model *once*, extract just the vectors we need (~1 MB),
cache that subset to Drive, and free the rest. Every later run loads in under a second.


In [ ]:
%%capture
# gensim <4.3.3 does `from scipy.linalg import triu`, which scipy>=1.13 removed --
# and Colab ships scipy 1.13+. Pin the version that dropped that import rather
# than downgrading scipy, which would break torch/sklearn.
!pip install -q "gensim>=4.3.3"


In [ ]:
import numpy as np, pickle, os, re

# ---- every string we will ever need a vector for ---------------------------
# Tags and context nouns, plus their individual words (needed for the
# compositional fallback when a multi-word phrase has no single vector).
def parts(s):
    return [w for w in re.split(r'[\s\-_/]+', s) if w]

need = set()
for t in lexicon:
    need.add(t); need.update(parts(t))
for r in rows:
    for n in r['context_nouns']:
        need.add(n); need.update(parts(n))

# GoogleNews is case-sensitive and joins phrases with '_'. Generate every
# surface form a token might plausibly be stored under, and keep whichever hits.
def variants(s):
    base = [s, s.lower(), s.title(), s.capitalize()]
    joined = [v.replace(' ', '_').replace('-', '_') for v in base]
    squashed = [v.replace(' ', '').replace('-', '') for v in base]
    seen, out = set(), []
    for v in base + joined + squashed:
        if v and v not in seen:
            seen.add(v); out.append(v)
    return out

probe = {v for s in need for v in variants(s)}
print(f'{len(need)} distinct strings -> {len(probe)} surface forms to probe')

# ---- load the subset cache, or build it from the full 3.6 GB model ---------
W2V_CACHE = f'{ROOT}/models_cache/w2v_googlenews_subset.pkl'
if os.path.exists(W2V_CACHE):
    VEC = pickle.load(open(W2V_CACHE, 'rb'))
    print(f'loaded cached subset: {len(VEC)} vectors from {W2V_CACHE}')
else:
    print('building subset cache (one-time ~3.6 GB download, a few minutes)...')
    import gensim.downloader as api
    kv = api.load('word2vec-google-news-300')          # 3M x 300
    VEC = {p: np.asarray(kv[p], dtype=np.float32) for p in probe if p in kv}
    del kv                                             # free ~3.6 GB
    import gc; gc.collect()
    pickle.dump(VEC, open(W2V_CACHE, 'wb'), protocol=4)
    print(f'cached {len(VEC)} vectors -> {W2V_CACHE} '
          f'({os.path.getsize(W2V_CACHE)/1e6:.1f} MB)')


## H2.1 — Algorithm 2: Noun-to-Tag similarity

Straight from the paper (p. 74). For a sentence's context nouns:

```
function NOUN-TO-TAG(ContextNouns, OSMTags)
    for each cn in ContextNouns:
        for each tag in OSMTags:
            sim[cn, tag] <- COMPUTESIMILARITY(cn, tag)
        normalize sim[cn] based on max and min values     <-- per noun, to [0,1]
    for each tag in OSMTags:
        score[tag] <- sum over cn of sim[cn, tag]
    return SORT(score)                                     <-- descending
```

**The per-noun min-max normalization is the load-bearing detail.** The paper: *"all results for a
single word are normalized such that the most similar OSM tag is set to 1.0 and the least similar
OSM tag is set to 0."* Its purpose is stated explicitly — *"each place must correspond to some
entity in the OSM database and each context word should equally contribute to finding the correct
tag."* Without it, a noun with globally high cosines would dominate the sum.

Verified against their worked example for *"This place has great coffee"* (p. 74): `coffee` is
closest to `cafe` 0.46 / `bakery` 0.41 / `icecream` 0.39; `place` to `streets` 0.25 / `building`
0.21 / `city` 0.20. After normalizing each noun to [0,1] and summing, their top-5 is
`cafe 1.59, bakehouse 1.40, restaurant 1.36, biergarten 1.33, bakery 1.27`.

### Two things the paper leaves unspecified

| Gap | Our documented choice |
|---|---|
| **Out-of-vocabulary terms** | GoogleNews is case-sensitive and joins phrases with `_`. Try every surface variant (`Spanish ham`, `spanish_ham`, `SpanishHam`, …); if none hits, back off to the **mean of the constituent word vectors**; if still nothing, drop the noun and count it. |
| **Multi-word context nouns** (`crab linguine`, `Michelin star`) | same variant-then-compose ladder — a phrase vector if one exists, otherwise the mean of its words. |

Both are reported as counts so the reproduction gap in H3 can be attributed honestly.


In [ ]:
from collections import Counter

OOV = Counter()          # strings we could not embed at all
COMPOSED = Counter()     # strings embedded by averaging their parts

def phrase_vec(s):
    """Vector for a tag or context noun, or None. Ladder: exact surface variant
    -> mean of constituent word vectors -> give up."""
    for v in variants(s):
        if v in VEC:
            return VEC[v]
    sub = [VEC[v] for w in parts(s) for v in variants(w) if v in VEC][:8]
    if sub:
        COMPOSED[s] += 1
        return np.mean(sub, axis=0)
    OOV[s] += 1
    return None

# ---- tag matrix, L2-normalized so cosine is a single dot product -----------
TAGS   = [t for t in lexicon if phrase_vec(t) is not None]
DROPPED_TAGS = [t for t in lexicon if t not in set(TAGS)]
TAG_M  = np.vstack([phrase_vec(t) for t in TAGS])
TAG_M /= np.linalg.norm(TAG_M, axis=1, keepdims=True) + 1e-9
TAG_IX = {t: i for i, t in enumerate(TAGS)}
print(f'tags embeddable: {len(TAGS)}/{len(lexicon)}'
      + (f'   dropped: {DROPPED_TAGS}' if DROPPED_TAGS else ''))


def noun_to_tag(context_nouns):
    """Algorithm 2 -> (ranked tag list, score vector aligned to TAGS, #nouns used)."""
    score = np.zeros(len(TAGS), dtype=np.float32)
    used = 0
    for cn in context_nouns:
        v = phrase_vec(cn)
        if v is None:
            continue
        sims = TAG_M @ (v / (np.linalg.norm(v) + 1e-9))
        lo, hi = sims.min(), sims.max()
        score += (sims - lo) / (hi - lo + 1e-9)      # per-noun min-max to [0,1]
        used += 1
    order = np.argsort(-score, kind='stable')
    return [TAGS[i] for i in order], score, used


# ---- sanity check against the paper's own worked example ------------------
ranked, sc, _ = noun_to_tag(['place', 'coffee'])
print('\npaper example "This place has great coffee" -> their top 5 was:')
print('   cafe 1.59, bakehouse 1.40, restaurant 1.36, biergarten 1.33, bakery 1.27')
print('ours:')
for t in ranked[:5]:
    print(f'   {t} {sc[TAG_IX[t]]:.2f}')


## H2.2 — Algorithm 3: Tag-to-Tag clustering

Their second algorithm (p. 75). Summing noun similarities often surfaces the right *category* but
not the exact tag; clustering re-ranks by asking which candidate tags are mutually related.

```
function TAG-TO-TAG(prunedTags, n)
    for each t_i in prunedTags:
        sim[i] <- [ COMPUTESIMILARITY(t_i, t_j) for t_j in prunedTags ]
        TopN(SORT(sim[i]), n)                    <-- keep only t_i's n closest neighbours
    sort prunedTags by number of occurrences across all sim lists
    tags with the same occurrence count form a cluster
    return sorted list of tag clusters
```

So each candidate votes for its `n` nearest neighbours; tags receiving more votes are more
"central" to the sentence's semantic neighbourhood and rank higher. Equal vote counts form one
cluster — an intentionally *coarse* ranking.

**Pruning.** Input is the **top-20** from Algorithm 2: *"We are using only the OSM tags appearing
at a position less than or equal to 20 from the noun-to-tag algorithm as input to the clustering
step."* This has a hard consequence the paper is explicit about — if the gold tag was not already
in Algorithm 2's top-20, clustering **can never recover it** (their `N/A` column, which equals the
`>20` column of the unclustered run). This is why clustering *lowers* GloVe and BERT scores even
while raising Word2Vec's.

**Tie-breaking — worst case.** The paper: *"In case multiple tags receive the same score and the
order is thus ambiguous, we follow a worst-case assumption as such that the correct tag is said to
be the last within this set."* This matters enormously here, because clustering produces large
tied groups by construction. We apply it to both algorithms: a gold tag's rank is the count of
tags scoring **greater than or equal to** it.

`n` (neighbours per tag) is not stated in the paper; we sweep it in H3.


In [ ]:
from itertools import groupby
from collections import Counter

PRUNE_K = 20        # paper: only tags at position <= 20 feed the clustering step

def tag_to_tag(ranked, n=4, prune_k=PRUNE_K):
    """Algorithm 3 -> (clusters, flat ranked list).

    clusters: list of lists, most-voted first. Ties inside a cluster are
    unordered by construction, which is exactly why worst-case ranking applies.
    """
    pruned = ranked[:prune_k]
    idx = [TAG_IX[t] for t in pruned]
    M = TAG_M[idx]                       # already L2-normalized -> dot == cosine
    S = M @ M.T
    np.fill_diagonal(S, -np.inf)         # a tag does not vote for itself

    votes = Counter({t: 0 for t in pruned})
    for i, t in enumerate(pruned):
        for j in np.argsort(-S[i])[:n]:
            votes[pruned[j]] += 1

    clusters, flat = [], []
    for _, grp in groupby(sorted(votes.items(), key=lambda kv: -kv[1]),
                          key=lambda kv: kv[1]):
        c = [t for t, _ in grp]
        clusters.append(c); flat.extend(c)
    return clusters, flat


def rank_of(gold, ranked, scores=None, clusters=None):
    """Worst-case 1-based rank of `gold`; len(ranked)+1 if absent (their 'N/A').

    Worst case = every tag tied with gold is placed ahead of it, per the paper.
    For Algorithm 2 ties are read off the score vector; for Algorithm 3 the
    tie group is the cluster.
    """
    if gold not in ranked:
        return len(ranked) + 1
    if clusters is not None:
        seen = 0
        for c in clusters:
            seen += len(c)
            if gold in c:
                return seen                     # last position within its cluster
    if scores is not None:
        return int((scores >= scores[TAG_IX[gold]]).sum())
    return ranked.index(gold) + 1


## H2.3 — Metrics

Their §4.2 definitions, plus the k values Dr. Yousaf asked for.

**Hits@k** — fraction of sentences whose gold tag lands at rank ≤ k. The paper motivates k
operationally: *"one would require k queries to the OSM database in order to find the correct
spatial entity... It is hence of practical interest to keep k as small as possible."*

> **Hits@3 is our headline metric** (supervisor directive): the correct tag inside the top 3 means
> the place is identified in **at most 3 OSM queries** — i.e. faster identification. The paper
> publishes only k = 1, 5, 10, 20; since we reproduce their pipeline we can evaluate any k, so we
> report **k = 1, 2, 3, 5, 10, 20** for *every* arm. Full range, nothing cherry-picked.

**nDCG@20** — `DCG = 1/log₂(r+1)` where `r` is the gold tag's position; ideal DCG = 1 (gold at
rank 1), so nDCG is just the mean DCG. Positions beyond 20 contribute 0, matching their top-k
convention.

**The baseline columns carry into every table from here on.** From H1: a majority-class rule scores
Hits@1 = 0.283, already above the paper's best published Hits@1 of 0.2264, and the (test-peeking)
frequency prior reaches **Hits@3 = 0.519**. Any Hits@3 result must be read against that 0.519 bar
or it says nothing.


In [ ]:
KS = (1, 2, 3, 5, 10, 20)          # 3 is the headline (supervisor directive)

def metrics(ranks, ks=KS):
    """ranks: 1-based worst-case rank of the gold tag per sentence."""
    r = np.asarray(ranks, dtype=float)
    out = {f'Hits@{k}': float((r <= k).mean()) for k in ks}
    out['nDCG@20'] = float(np.where(r <= 20, 1.0 / np.log2(r + 1), 0.0).mean())
    out['MedRank'] = float(np.median(r))
    return out

# --- self-test: a perfect ranker scores 1.0 everywhere, a rank-4 ranker doesn't
assert metrics([1, 1, 1])['Hits@1'] == 1.0
assert metrics([4, 4, 4])['Hits@3'] == 0.0 and metrics([4, 4, 4])['Hits@5'] == 1.0
assert abs(metrics([1])['nDCG@20'] - 1.0) < 1e-9
assert metrics([21])['nDCG@20'] == 0.0          # beyond top-20 contributes nothing

def show(name, m, ks=KS):
    print(f'{name:<34}' + ''.join(f'{m[f"Hits@{k}"]:>9.4f}' for k in ks)
          + f'{m["nDCG@20"]:>10.4f}{m["MedRank"]:>9.1f}')

def header(ks=KS):
    print(f'{"":34}' + ''.join(f'{"H@"+str(k):>9}' for k in ks)
          + f'{"nDCG@20":>10}{"MedR":>9}')

# ---- the two priors, carried into every results table ----------------------
dist = Counter(r['gold_tag'] for r in rows)
by_freq = [t for t, _ in dist.most_common()]
maj = by_freq[0]
BASELINES = {
    'majority class ("%s")' % maj:
        metrics([1 if r['gold_tag'] == maj else len(TAGS) + 1 for r in rows]),
    'frequency prior (test-peeking)':
        metrics([by_freq.index(r['gold_tag']) + 1 for r in rows]),
}
header()
for k, v in BASELINES.items():
    show(k, v)
print('\nYousaf 2023 published (W2V Clustered): H@1 0.2264  H@5 0.5283  '
      'H@10 0.6509  H@20 0.7075  nDCG@20 0.4442   [no H@3 published]')


---

# H3 — Reproduce Yousaf 2023, Table 4 & 5

**The credibility gate.** We run their Algorithm 2 and Algorithm 3 on *their* context nouns, over
*their* tag lexicon, scored with *their* metrics, and compare to their published numbers. Nothing of
ours is involved.

If this does not land close, every later claim ("adjectives improve on the baseline") is worthless —
we would just be comparing against a broken reimplementation rather than against their method.

### Targets

| Model | Hits@1 | Hits@5 | Hits@10 | Hits@20 | nDCG@20 |
|---|---|---|---|---|---|
| W2V | 0.1604 | 0.4717 | 0.6509 | 0.7075 | 0.3968 |
| **W2V Clustered** | **0.2264** | **0.5283** | 0.6509 | 0.7075 | **0.4442** |

### What we expect to differ, and why

The paper leaves several implementation details unstated, so an *exact* match would be surprising —
what we need is agreement in level and in the plain-vs-clustered ordering:

- **`n`** (neighbours each tag votes for in Algorithm 3) is never given — we sweep it below and
  report the sweep rather than silently picking the flattering value.
- **Lexicon dedup** — we collapsed 10 plural duplicates (H1.4); they appear not to have. This shifts
  every per-noun min-max normalization slightly, which is exactly the ~+0.05 offset already visible
  in the worked example above.
- **OOV / multi-word handling** — our documented ladder (H2.1) is one reasonable choice among several.

Deltas are reported, not tuned away.


In [ ]:
PUBLISHED = {
    'W2V':           {'Hits@1': .1604, 'Hits@5': .4717, 'Hits@10': .6509,
                      'Hits@20': .7075, 'nDCG@20': .3968},
    'W2V Clustered': {'Hits@1': .2264, 'Hits@5': .5283, 'Hits@10': .6509,
                      'Hits@20': .7075, 'nDCG@20': .4442},
}

# ---- Algorithm 2 over all 106 rows, on the AUTHORS' context nouns ----------
alg2 = []                       # (ranked, scores) per row, reused by the sweep
ranks_plain = []
for r in rows:
    ranked, sc, used = noun_to_tag(r['context_nouns'])
    alg2.append((ranked, sc))
    ranks_plain.append(rank_of(r['gold_tag'], ranked, scores=sc))

res_plain = metrics(ranks_plain)

print(f'embedding coverage: {len(COMPOSED)} phrases composed from parts, '
      f'{len(OOV)} unembeddable {sorted(OOV)[:8]}')
print()
header()
show('W2V  (ours)', res_plain)
show('W2V  (published)', {**PUBLISHED['W2V'], 'Hits@2': float("nan"),
                          'Hits@3': float("nan"), 'MedRank': float("nan")})


In [ ]:
# `n` (neighbours each pruned tag votes for) is not stated in the paper -> sweep it.
print('Algorithm 3 sweep over n  (prune_k = %d)\n' % PRUNE_K)
header()
sweep = {}
for n in range(2, 11):
    rk = []
    for r, (ranked, sc) in zip(rows, alg2):
        clusters, flat = tag_to_tag(ranked, n=n)
        rk.append(rank_of(r['gold_tag'], flat, clusters=clusters))
    sweep[n] = metrics(rk)
    show(f'W2V Clustered  n={n}', sweep[n])

show('W2V Clustered  (published)',
     {**PUBLISHED['W2V Clustered'], 'Hits@2': float('nan'),
      'Hits@3': float('nan'), 'MedRank': float('nan')})

# Pick n by closeness to the PUBLISHED numbers -- not by what flatters us.
def dist_to_published(m):
    return sum(abs(m[k] - v) for k, v in PUBLISHED['W2V Clustered'].items())

BEST_N = min(sweep, key=lambda n: dist_to_published(sweep[n]))
print(f'\nclosest to published: n={BEST_N} '
      f'(mean abs deviation {dist_to_published(sweep[BEST_N])/5:.4f} across their 5 reported cells)')


### H3.1 — Resolving an ambiguity in Algorithm 3

Our clustering *lowers* scores, but the paper reports it *raising* W2V (Hits@1 0.1604 → 0.2264).
Before accepting that, check whether we read the algorithm the way they implemented it. The text is
genuinely ambiguous about **where each tag's neighbours are drawn from**:

> *"by taking the n most related other OSM tags for each OSM tag… Given a matrix of n related tags
> for each of the m pruned OSM tags, we simply count the number of times each OSM tag is represented."*

| Reading | Neighbours drawn from | Consequence |
|---|---|---|
| **A** (what we built) | the 20 pruned tags only | every vote lands on a pruned tag; total votes = 20·n exactly |
| **B** | the **full 227-tag lexicon**, then count only how often *pruned* tags appear | votes leaking outside the pruned set are discarded, so totals are < 20·n |

Their own worked example decides it. 12 pruned tags, and the vote counts they list are
`picnic-table 6; garden/food-court/fast-food 5; house/restaurant/houseboat/drinking-water/bird-hide 4;
cafe/farm/bar 3` — summing to **50 votes over 12 tags**.

- Reading A requires `12·n = 50` → `n = 4.17`, **not an integer**.
- Reading B with `n = 5` casts `12·5 = 60` votes, of which 50 land inside the pruned set and 10 leak
  out — **consistent**.

So reading B with n≈5 is the better fit to their description. We implement both and let the
reproduction decide, rather than picking the one that flatters us.


In [ ]:
def tag_to_tag_B(ranked, n=5, prune_k=PRUNE_K):
    """Reading B: each pruned tag's n nearest neighbours are taken from the FULL
    lexicon; votes landing outside the pruned set are discarded."""
    pruned = ranked[:prune_k]
    pset = set(pruned)
    idx = [TAG_IX[t] for t in pruned]
    S = TAG_M[idx] @ TAG_M.T                 # 20 x 227
    for i, t in enumerate(pruned):
        S[i, TAG_IX[t]] = -np.inf            # no self-vote

    votes = Counter({t: 0 for t in pruned})
    for i in range(len(pruned)):
        for j in np.argsort(-S[i])[:n]:
            t = TAGS[j]
            if t in pset:
                votes[t] += 1

    clusters, flat = [], []
    for _, grp in groupby(sorted(votes.items(), key=lambda kv: -kv[1]),
                          key=lambda kv: kv[1]):
        c = [t for t, _ in grp]
        clusters.append(c); flat.extend(c)
    return clusters, flat


print('Reading B sweep\n')
header()
sweepB = {}
for n in range(3, 13):
    rk = []
    for r, (ranked, sc) in zip(rows, alg2):
        clusters, flat = tag_to_tag_B(ranked, n=n)
        rk.append(rank_of(r['gold_tag'], flat, clusters=clusters))
    sweepB[n] = metrics(rk)
    show(f'W2V Clustered-B  n={n}', sweepB[n])

show('W2V Clustered  (published)',
     {**PUBLISHED['W2V Clustered'], 'Hits@2': float('nan'),
      'Hits@3': float('nan'), 'MedRank': float('nan')})

bestB = min(sweepB, key=lambda n: dist_to_published(sweepB[n]))
print(f'\nreading A best: n={BEST_N}, mean abs dev '
      f'{dist_to_published(sweep[BEST_N])/5:.4f}')
print(f'reading B best: n={bestB}, mean abs dev '
      f'{dist_to_published(sweepB[bestB])/5:.4f}')


In [ ]:
# Reading B reproduces the published behaviour (clustering IMPROVES over plain,
# 0.1792 -> 0.2358 Hits@1) whereas reading A inverted it. Lock B in.
# n chosen by lowest mean absolute deviation across ALL FIVE published cells,
# not by any single flattering cell (n=3 nails Hits@1 at 0.2264 exactly, but
# n=4 is closer overall).
CLUSTER_N = bestB
tag_to_tag_final = tag_to_tag_B

ranks_clust = []
for r, (ranked, sc) in zip(rows, alg2):
    clusters, flat = tag_to_tag_final(ranked, n=CLUSTER_N)
    ranks_clust.append(rank_of(r['gold_tag'], flat, clusters=clusters))
res_clust = metrics(ranks_clust)

print(f'{"H3 REPRODUCTION  (Algorithm 3 = reading B, n=%d)" % CLUSTER_N:-^100}\n')
header()
for k, v in BASELINES.items():
    show('  baseline: ' + k, v)
print()
show('W2V             ours', res_plain)
show('W2V             published', {**PUBLISHED['W2V'], 'Hits@2': float('nan'),
                                   'Hits@3': float('nan'), 'MedRank': float('nan')})
show('W2V Clustered   ours', res_clust)
show('W2V Clustered   published', {**PUBLISHED['W2V Clustered'], 'Hits@2': float('nan'),
                                   'Hits@3': float('nan'), 'MedRank': float('nan')})

print('\ndeltas vs published (ours - theirs):')
for name, ours, pub in (('W2V', res_plain, PUBLISHED['W2V']),
                        ('W2V Clustered', res_clust, PUBLISHED['W2V Clustered'])):
    d = {k: ours[k] - v for k, v in pub.items()}
    print(f'  {name:<15}' + '  '.join(f'{k} {v:+.4f}' for k, v in d.items()))
    print(f'  {"":<15}mean |delta| = {sum(abs(v) for v in d.values())/len(d):.4f}')

json.dump({'reading': 'B', 'cluster_n': CLUSTER_N,
           'W2V': res_plain, 'W2V_clustered': res_clust,
           'published': PUBLISHED, 'baselines': BASELINES,
           'ranks_plain': ranks_plain, 'ranks_clustered': ranks_clust},
          open(f'{ROOT}/results_h/h3_reproduction.json', 'w'), indent=1)
print(f'\nwrote {ROOT}/results_h/h3_reproduction.json')

print(f'\n>>> HITS@3 BAR FOR OUR METHOD <<<')
print(f'    reproduced baseline  W2V Clustered  Hits@3 = {res_clust["Hits@3"]:.4f}')
print(f'    frequency prior (test-peeking)      Hits@3 = {BASELINES["frequency prior (test-peeking)"]["Hits@3"]:.4f}')
print(f'    -> H5 must clear BOTH to make the supervisor\'s Hits@3 claim stand.')


---

# H4 — Fix `adjective_extractor.py`, then run it on the 106 sentences

This module is the thesis contribution: a spaCy dependency-parse implementation of the paper's
Algorithm 1 (named / unnamed-via-OSM-category / context nouns), **extended** with the adjective layer
footnote 6 discards. It emits `amod:` pairs for FP-Growth and `tag_hints` mapping adjectives straight
to OSM keys (`italian` → `cuisine=italian`, `brick` → `building:material=brick`).

Three bugs are fixed here; one is deliberately deferred.

### Fix 1 — participial adjectives get a verb lemma

`en_core_web_trf` lemmatizes participial adjectives as verbs:

| phrase | current item | should be |
|---|---|---|
| `winding streets` | `amod:wind_street` | `amod:winding_street` |
| `sprawling city` | `amod:sprawl_city` | `amod:sprawling_city` |
| `bustling market` | `amod:bustle_market` | `amod:bustling_market` |

Morphologically correct for a verb, wrong for a descriptive pattern — and it pollutes the mined
vocabulary corpus-wide. Fix: keep the **surface form** when the fine-grained tag is `VBG`/`VBN`,
lemma otherwise (so `narrower` → `narrow` still works).

### Fix 2 — `old` fires `historic=*` on everything

`ADJ_MODIFIER["old"] = ("historic", "*")` currently emits a heritage hint for `old street`,
`old shop`, `old friend`. Gate the heritage adjectives (`old`, `ancient`, `historic`, `colonial`,
`medieval`, `mughal`) so they only fire when the **head noun is itself a heritage-capable type**
(`fort`, `castle`, `church`, `town`, …).

### Fix 3 — the logic is implemented twice

Block D Cell 8 re-inlines the whole body of `interpret()` instead of calling it, so every fix has to
be mirrored in two places or the corpus and the eval silently diverge. Refactor into one
implementation with three entry points: `interpret_doc(doc)` holds the logic, `interpret(text)`
wraps it, and `pipe(texts)` batches through `nlp.pipe` for corpus-scale runs. Block D's inline copy
becomes deletable.

### Deferred — wiring the Urdu gazetteer

`PlaceInterpreter(gazetteer=…)` accepts a callable but Block D never passes one, so `_is_named`
relies purely on spaCy NER. The 264k `name:ur` gazetteer from Block B would improve this — **but only
for Pakistani place names**, and these 106 sentences are London/Berlin/Rome food venues. It cannot
affect H4/H5. Deferred to Block I, where the Pakistan corpus is actually used.


In [ ]:
import shutil, datetime

SRC = f'{ROOT}/adjective_extractor.py'
BAK = f'{ROOT}/adjective_extractor_pre_h4_{datetime.date.today():%Y%m%d}.py.bak'
if not os.path.exists(BAK):
    shutil.copy(SRC, BAK)
    print(f'backed up original -> {os.path.basename(BAK)}')

src = open(SRC, encoding='utf-8').read()

def patch(text, old, new, label):
    """Replace exactly once, or fail loudly -- never silently no-op."""
    if new in text and old not in text:
        print(f'  = {label} (already applied)')
        return text
    n = text.count(old)
    assert n == 1, f'{label}: expected 1 match, found {n}'
    print(f'  + {label}')
    return text.replace(old, new)

print('patching adjective_extractor.py')

# ---- Fix 1 + 2: helper + heritage gate, inserted after PLACE_HINT_NOUNS -----
src = patch(src,
'PLACE_HINT_NOUNS = {"place", "spot", "location", "area", "site", "position"}',
'''PLACE_HINT_NOUNS = {"place", "spot", "location", "area", "site", "position"}

# Heritage adjectives are only evidence of historic=* when the thing they modify
# can BE historic. Without this gate "old street" / "old friend" emit a heritage
# tag hint, which is noise: `old` is one of the commonest adjectives in the corpus.
HERITAGE_NOUNS = {
    "fort", "fortress", "citadel", "castle", "palace", "tomb", "monument", "ruin",
    "church", "mosque", "temple", "shrine", "cathedral", "chapel", "abbey",
    "monastery", "tower", "gate", "wall", "bridge", "mill", "manor", "building",
    "house", "quarter", "city", "town", "village", "district", "museum", "site",
}
HERITAGE_ADJ = {"old", "ancient", "historic", "colonial", "medieval", "mughal"}


def adj_form(tok):
    """Surface form to use for an adjective token.

    spaCy lemmatizes participial adjectives with their VERB lemma:
    'winding streets' -> 'wind', 'sprawling' -> 'sprawl', 'bustling' -> 'bustle'.
    Correct verb morphology, wrong descriptor -- the adjective is 'winding'. Keep
    the surface form for participles (VBG/VBN); lemma otherwise, so comparatives
    still normalize ('narrower' -> 'narrow').
    """
    if tok.tag_ in ("VBG", "VBN"):
        return tok.text.lower()
    return tok.lemma_.lower()''',
'fix 1+2: adj_form() helper and heritage gate')

# ---- Fix 1 applied at both call sites --------------------------------------
src = patch(src,
'            adj_lemmas = [a.lemma_.lower() for a in adjs]',
'            adj_lemmas = [adj_form(a) for a in adjs]',
'fix 1: amod adjectives use adj_form()')

src = patch(src,
'''            for c in phrase_toks[:-1]:       # noun-noun compounds
                cl = c.lemma_.lower()''',
'''            for c in phrase_toks[:-1]:       # noun-noun compounds
                cl = adj_form(c) if c.pos_ == "ADJ" else c.lemma_.lower()''',
'fix 1: compound-as-adjective uses adj_form()')

# ---- Fix 2 applied at the hint-emission site -------------------------------
src = patch(src,
'''                elif a in ADJ_MODIFIER:
                    k, v = ADJ_MODIFIER[a]
                    hints.append({"key": k, "value": v,
                                  "evidence": self._evidence(a, phrase)})''',
'''                elif a in ADJ_MODIFIER:
                    # heritage adjectives need a heritage-capable head noun
                    if a in HERITAGE_ADJ and lemma not in HERITAGE_NOUNS:
                        continue
                    k, v = ADJ_MODIFIER[a]
                    hints.append({"key": k, "value": v,
                                  "evidence": self._evidence(a, phrase)})''',
'fix 2: gate heritage adjectives on the head noun')

# ---- Fix 3: single implementation, three entry points ----------------------
src = patch(src,
'''    def interpret(self, sentence: str) -> Interpretation:
        doc = self.nlp(sentence)''',
'''    def interpret(self, sentence: str) -> Interpretation:
        """Single sentence -> Interpretation."""
        return self.interpret_doc(self.nlp(sentence))

    def pipe(self, texts, batch_size: int = 128):
        """Batched interpretation over many texts -- one spaCy pass per batch.

        Exists so corpus-scale callers (Block D) reuse THIS logic instead of
        re-inlining it; two copies drift the moment either is fixed.
        """
        for doc in self.nlp.pipe(texts, batch_size=batch_size):
            yield self.interpret_doc(doc)

    def interpret_doc(self, doc) -> Interpretation:
        """The actual Algorithm-1-plus-adjectives implementation."""
        sentence = doc.text''',
'fix 3: interpret_doc() + pipe(), interpret() delegates')

open(SRC, 'w', encoding='utf-8').write(src)
print(f'\nwrote {SRC}  ({len(src):,} bytes)')


In [ ]:
%%capture --no-stderr
import torch
# en_core_web_trf gives the best parses (Block C+D used it for the 1M corpus, so
# H4/H5 must match that to stay comparable). 106 sentences run fine on CPU.
!pip install -q "spacy>=3.7,<3.9" spacy-transformers
!python -m spacy download en_core_web_trf -q


In [ ]:
import sys, importlib, spacy, torch
sys.path.insert(0, ROOT)
import adjective_extractor as ax
importlib.reload(ax)                     # pick up the patched file

if torch.cuda.is_available():
    spacy.require_gpu()
MODEL = 'en_core_web_trf'
pi = ax.PlaceInterpreter(model=MODEL)
print(f'{MODEL} loaded  (GPU={torch.cuda.is_available()})')

# ---- verify each fix actually changed behaviour ---------------------------
print('\n--- fix 1: participial adjectives keep their surface form ---')
r = pi.interpret('The narrow winding streets of Raja Bazaar are lined with old brick shops.')
amods = [p for p in r.pairs if p[2] == 'amod']
print('  pairs:', amods)
assert any(a == 'winding' for a, _, _ in amods), 'winding should not lemmatize to wind'
print('  OK: "winding" survives (was "wind" before)')

print('\n--- fix 2: heritage gate ---')
for s, expect in [('The ancient fort overlooks the valley.', True),
                  ('We walked down the old street to the shop.', False)]:
    h = pi.interpret(s).tag_hints
    got = any(x['key'] == 'historic' for x in h)
    print(f'  historic hint={got:<5}  expected={expect:<5}  {s}')
    assert got == expect
print('  OK: historic=* only fires on heritage-capable nouns')

print('\n--- fix 3: one implementation, three entry points ---')
one = pi.interpret('This Persian place makes the best bread in the city.')
batched = list(pi.pipe(['This Persian place makes the best bread in the city.']))[0]
assert one.transaction == batched.transaction and one.tag_hints == batched.tag_hints
print('  OK: interpret() and pipe() give identical output')
print('  tag_hints:', one.tag_hints)
print('  transaction:', one.transaction)


In [ ]:
from tqdm.auto import tqdm

interps = list(tqdm(pi.pipe([r['sentence'] for r in rows], batch_size=32),
                    total=len(rows), desc='extracting'))

def _stem(w):
    """Crude singularizer, ONLY for comparing our lemmatized nouns against the
    authors' unlemmatized ones ('owls' vs 'owl', 'dishes' vs 'dish')."""
    w = w.lower()
    for suf, rep in (('ies', 'y'), ('sses', 'ss'), ('shes', 'sh'), ('ches', 'ch'),
                     ('es', ''), ('s', '')):
        if len(w) > 3 and w.endswith(suf):
            return w[:-len(suf)] + rep
    return w

def words(strings):
    return {_stem(w) for s in strings for w in re.split(r'[\s\-_/]+', s) if len(w) > 1}

n_adj, n_hint, recall, extra_adj = [], [], [], []
for r, it in zip(rows, interps):
    ours_nouns = [e['head_lemma'] for e in it.entities] + list(it.context_nouns)
    adjs = sorted({a for e in it.entities for a in e['adjectives']})
    theirs, ours = words(r['context_nouns']), words(ours_nouns)
    r['ours_nouns']  = sorted(set(ours_nouns))
    r['adjectives']  = adjs
    r['tag_hints']   = it.tag_hints
    r['transaction'] = it.transaction
    n_adj.append(len(adjs))
    n_hint.append(len(it.tag_hints))
    recall.append(len(theirs & ours) / len(theirs) if theirs else np.nan)
    extra_adj.append(len([a for a in adjs if _stem(a) not in theirs]))

recall = np.array(recall, dtype=float)
n_adj, n_hint, extra_adj = map(np.array, (n_adj, n_hint, extra_adj))

print(f'{"H4 EXTRACTION ON THE 106 SENTENCES":-^78}\n')
print(f'recall of the authors\' context nouns : {np.nanmean(recall):.1%} '
      f'(median {np.nanmedian(recall):.0%}) [stem-matched]')
print(f'   rows recovering >=80% of their nouns: '
      f'{(recall >= 0.8).sum()}/{len(rows)}\n')
print(f'adjectives extracted   : {n_adj.sum()} total, mean {n_adj.mean():.2f}/sentence')
print(f'   sentences with >=1  : {(n_adj > 0).sum()}/{len(rows)}  ({(n_adj > 0).mean():.1%})')
print(f'   NOT in their nouns  : {extra_adj.sum()} total  <-- the signal footnote 6 discards')
print(f'   sentences with >=1  : {(extra_adj > 0).sum()}/{len(rows)}  ({(extra_adj > 0).mean():.1%})')
print(f'\ntag_hints (adj -> OSM key) : {n_hint.sum()} total')
print(f'   sentences with >=1     : {(n_hint > 0).sum()}/{len(rows)}  ({(n_hint > 0).mean():.1%})')

missed = Counter()
for r, it in zip(rows, interps):
    missed.update(words(r['context_nouns']) - words(r['ours_nouns']))
print('\nauthors\' noun-words still missed (after stem-matching):')
for w, c in missed.most_common(12):
    print(f'   {c:>3}  {w}')

print('\nmost frequent extracted adjectives:')
for a, c in Counter(a for r in rows for a in r['adjectives']).most_common(15):
    print(f'   {c:>3}  {a}')

print('\nexample rows -- what we add over their nouns:')
for i in (3, 20, 64, 106):
    r = rows[i - 1]
    print(f"  [{r['row']}] gold={r['gold_tag']}")
    print(f"      theirs : {r['context_nouns']}")
    print(f"      + adj  : {r['adjectives']}")
    if r['tag_hints']:
        print(f"      + hint : {[(h['key'], h['value']) for h in r['tag_hints']]}")

json.dump(rows, open(f'{ROOT}/eval/evalset_106_extracted.json', 'w'),
          indent=1, ensure_ascii=False, default=str)
print(f'\nwrote {ROOT}/eval/evalset_106_extracted.json')


---

# H5 — Adjective extension, ablated

**The RQ1 result.** Every arm below starts from the *same* baseline as H3 — the authors' own context
nouns, their Algorithm 2/3, their metrics — and changes exactly one thing. So any delta is
attributable to that one change and nothing else.

| # | Arm | Change from baseline |
|---|---|---|
| 0 | **baseline** | their nouns only (identical to H3) |
| 1 | + all adjectives | append every extracted adjective as a context term |
| 2 | + abstract-noun removal | drop `place / spot / location / area / site / thing` — *the authors' own stated future work* |
| 3 | + tag-hint boost | adjective→OSM-key hints (`italian`→`cuisine`) force the implied tag to rank 1 |
| 4 | 2 + 3 | |
| 5 | full | 2 + 3 + all adjectives |

Reported for **W2V** and **W2V-Clustered** (reading B, n=4), plus a **DistilBERT** backend to sit
against their BERT/DeBERTa row (nDCG 0.2911). Every table carries the two H1 baseline rows.

**What we already expect** (H4 note): the extracted adjectives are mostly vague (`good` ×14,
`beautiful` ×10, `great` ×9). Arm 1 will likely *dilute*. That is not a failure — it is the
argument for Block I, where FP-Growth + Elastic Net *select* which adjective patterns carry tag
signal instead of dumping all of them in.


In [ ]:
# ============================================================================
# H5 RESTORE  — reload H1-H4 state from Drive. Skips re-running H1-H4.
# ============================================================================
from google.colab import drive; drive.mount('/content/drive')
import os, json, re, pickle, numpy as np
from collections import Counter
from itertools import groupby

ROOT = '/content/drive/MyDrive/geoproject'; EVAL = f'{ROOT}/eval'
rows = json.load(open(f'{EVAL}/evalset_106_extracted.json'))          # H1 + H4 output
_d   = json.load(open(f'{EVAL}/evalset_106.json'))
lexicon, singular, lex_norm = _d['lexicon'], _d['plural_map'], set(_d['lexicon']) | set(_d['plural_map'])
VEC  = pickle.load(open(f'{ROOT}/models_cache/w2v_googlenews_subset.pkl', 'rb'))
print(f'{len(rows)} rows, {len(lexicon)} tags, {len(VEC)} w2v vectors')

# ---- locked constants (see checkpoint / H3) --------------------------------
PRUNE_K, CLUSTER_N = 20, 4
KS = (1, 2, 3, 5, 10, 20)

# ---- H2/H3 functions, re-defined verbatim ----------------------------------
def parts(s):
    return [w for w in re.split(r'[\s\-_/]+', s) if w]

def variants(s):
    base = [s, s.lower(), s.title(), s.capitalize()]
    out, seen = [], set()
    for v in base + [b.replace(' ', '_').replace('-', '_') for b in base] \
                  + [b.replace(' ', '').replace('-', '') for b in base]:
        if v and v not in seen:
            seen.add(v); out.append(v)
    return out

OOV, COMPOSED = Counter(), Counter()
def phrase_vec(s):
    for v in variants(s):
        if v in VEC:
            return VEC[v]
    sub = [VEC[v] for w in parts(s) for v in variants(w) if v in VEC][:8]
    if sub:
        COMPOSED[s] += 1; return np.mean(sub, axis=0)
    OOV[s] += 1; return None

TAGS  = [t for t in lexicon if phrase_vec(t) is not None]
TAG_M = np.vstack([phrase_vec(t) for t in TAGS])
TAG_M /= np.linalg.norm(TAG_M, axis=1, keepdims=True) + 1e-9
TAG_IX = {t: i for i, t in enumerate(TAGS)}

def noun_to_tag(terms, tag_m=None, tag_ix=None, embed=phrase_vec):
    tag_m = TAG_M if tag_m is None else tag_m
    tag_ix = TAG_IX if tag_ix is None else tag_ix
    score = np.zeros(tag_m.shape[0], dtype=np.float32)
    for cn in terms:
        v = embed(cn)
        if v is None:
            continue
        sims = tag_m @ (v / (np.linalg.norm(v) + 1e-9))
        score += (sims - sims.min()) / (sims.max() - sims.min() + 1e-9)
    order = np.argsort(-score, kind='stable')
    return [TAGS[i] for i in order], score

def tag_to_tag_B(ranked, tag_m=None, tag_ix=None, n=CLUSTER_N, prune_k=PRUNE_K):
    tag_m = TAG_M if tag_m is None else tag_m
    tag_ix = TAG_IX if tag_ix is None else tag_ix
    pruned = ranked[:prune_k]; pset = set(pruned)
    S = tag_m[[tag_ix[t] for t in pruned]] @ tag_m.T
    for i, t in enumerate(pruned):
        S[i, tag_ix[t]] = -np.inf
    votes = Counter({t: 0 for t in pruned})
    names = list(tag_ix)                              # index -> tag, order of tag_ix
    for i in range(len(pruned)):
        for j in np.argsort(-S[i])[:n]:
            if names[j] in pset:
                votes[names[j]] += 1
    clusters, flat = [], []
    for _, grp in groupby(sorted(votes.items(), key=lambda kv: -kv[1]), key=lambda kv: kv[1]):
        c = [t for t, _ in grp]; clusters.append(c); flat.extend(c)
    return clusters, flat

def rank_of(gold, ranked, scores=None, clusters=None, tag_ix=None):
    tag_ix = TAG_IX if tag_ix is None else tag_ix
    if gold not in ranked:
        return len(ranked) + 1
    if clusters is not None:
        seen = 0
        for c in clusters:
            seen += len(c)
            if gold in c:
                return seen
    if scores is not None:
        return int((scores >= scores[tag_ix[gold]]).sum())
    return ranked.index(gold) + 1

def metrics(ranks, ks=KS):
    r = np.asarray(ranks, dtype=float)
    out = {f'Hits@{k}': float((r <= k).mean()) for k in ks}
    out['nDCG@20'] = float(np.where(r <= 20, 1.0 / np.log2(r + 1), 0.0).mean())
    out['MedRank'] = float(np.median(r))
    return out

def header(ks=KS):
    print(f'{"":30}' + ''.join(f'{"H@"+str(k):>8}' for k in ks) + f'{"nDCG":>9}{"MedR":>7}')
def show(name, m, ks=KS):
    print(f'{name:<30}' + ''.join(f'{m[f"Hits@{k}"]:>8.4f}' for k in ks)
          + f'{m["nDCG@20"]:>9.4f}{m["MedRank"]:>7.1f}')

_dist = Counter(r['gold_tag'] for r in rows)
_by_freq = [t for t, _ in _dist.most_common()]
BASELINES = {
    f'baseline: majority ("{_by_freq[0]}")':
        metrics([1 if r['gold_tag'] == _by_freq[0] else len(TAGS) + 1 for r in rows]),
    'baseline: frequency prior (peeks)':
        metrics([_by_freq.index(r['gold_tag']) + 1 for r in rows]),
}

# sanity: reproduce the H3 W2V-clustered number from the reloaded state
_r = []
for r in rows:
    ranked, sc = noun_to_tag(r['context_nouns'])
    cl, flat = tag_to_tag_B(ranked)
    _r.append(rank_of(r['gold_tag'], flat, clusters=cl))
_m = metrics(_r)
print(f'\nH3 W2V-Clustered reproduced from reload: '
      f'H@1={_m["Hits@1"]:.4f} H@3={_m["Hits@3"]:.4f} nDCG={_m["nDCG@20"]:.4f}')
assert abs(_m['Hits@1'] - 0.2358) < 1e-3, 'state reload mismatch vs H3'
print('restore OK')


In [ ]:
# ---- ablation arms: each maps a row -> (context terms, forced-to-front tags) --
ABSTRACT = {'place', 'spot', 'location', 'area', 'site', 'position', 'thing',
            'kind', 'sort', 'lot', 'bit', 'way', 'day', 'time'}

# A tag_hint is {key,value} like cuisine=italian. This dataset's gold labels are
# place *types* (cafe, restaurant), not attributes, so a hint can only narrow the
# candidate set, not name the tag. Minimal, defensible mapping:
HINT_TAGS = {
    'cuisine':            ['restaurant', 'cafe', 'fast-food', 'pub', 'bar'],
    'diet:vegan':         ['restaurant', 'cafe', 'fast-food'],
    'diet:vegetarian':    ['restaurant', 'cafe', 'fast-food'],
    'diet:halal':         ['restaurant', 'fast-food'],
    'historic':           ['castle', 'fort', 'monument', 'attraction', 'museum', 'ruins'],
    'building:material':  ['building', 'house', 'church'],
}
def hint_tags(row):
    out = []
    for h in row['tag_hints']:
        out += [t for t in HINT_TAGS.get(h['key'], []) if t in TAG_IX]
        if h['value'] in TAG_IX:                       # value is itself a tag
            out.append(h['value'])
    return list(dict.fromkeys(out))

def arm_terms(row, drop_abstract=False, add_adj=False):
    terms = [t for t in row['context_nouns']
             if not (drop_abstract and t.lower() in ABSTRACT)]
    if add_adj:
        terms = terms + row['adjectives']
    return terms

ARMS = {
    '0 baseline (their nouns)':        dict(),
    '1 + all adjectives':              dict(add_adj=True),
    '2 + abstract-noun removal':       dict(drop_abstract=True),
    '3 + tag-hint boost':              dict(boost=True),
    '4 = 2 + 3':                       dict(drop_abstract=True, boost=True),
    '5 full (2+3+adj)':                dict(drop_abstract=True, add_adj=True, boost=True),
}

def run_arm(drop_abstract=False, add_adj=False, boost=False, clustered=False):
    ranks = []
    for r in rows:
        ranked, sc = noun_to_tag(arm_terms(r, drop_abstract, add_adj))
        if clustered:
            cl, flat = tag_to_tag_B(ranked)
            if boost:
                front = [t for t in hint_tags(r) if t in flat]
                flat = front + [t for t in flat if t not in front]
                cl = ([front] if front else []) + [[t for t in c if t not in front] for c in cl]
            ranks.append(rank_of(r['gold_tag'], flat, clusters=cl))
        else:
            if boost:
                front = hint_tags(r)
                ranked = front + [t for t in ranked if t not in front]
                # worst-case: gold tied with the boosted block if it's in it
                rk = (ranked.index(r['gold_tag']) + 1) if r['gold_tag'] in ranked else len(ranked) + 1
                ranks.append(rk)
            else:
                ranks.append(rank_of(r['gold_tag'], ranked, scores=sc))
    return metrics(ranks)

n_boostable = sum(bool(hint_tags(r)) for r in rows)
print(f'tag-hint boost fires on {n_boostable}/{len(rows)} rows '
      f'({n_boostable/len(rows):.1%})\n')

for title, clustered in [('W2V  (Algorithm 2 only)', False),
                         ('W2V-Clustered  (Algorithm 3, reading B, n=4)', True)]:
    print(f'{title:-^88}')
    header()
    for k, v in BASELINES.items():
        show(k, v)
    print()
    res = {}
    for name, kw in ARMS.items():
        res[name] = run_arm(clustered=clustered, **kw)
        show(name, res[name])
    d0 = res['0 baseline (their nouns)']
    print('\n  deltas vs arm 0:')
    for name in list(ARMS)[1:]:
        dd = {k: res[name][k] - d0[k] for k in ('Hits@1', 'Hits@3', 'Hits@5', 'nDCG@20')}
        print(f'    {name:<28}' + '  '.join(f'{k} {v:+.4f}' for k, v in dd.items()))
    print()
    json.dump({n: m for n, m in res.items()},
              open(f'{ROOT}/results_h/h5_{"clustered" if clustered else "plain"}_w2v.json', 'w'),
              indent=1)


## H5.1 — DistilBERT embedding backend

Swap the W2V vectors for **mean-pooled `distilbert-base-uncased`** embeddings of each word/tag,
same Algorithm 2/3 otherwise. This is the head-to-head against the paper's BERT row (they used
DeBERTa-large via BERTScore, nDCG@20 **0.2911** — their *worst* embedding).

Only the winning arms are run here (baseline + abstract-noun removal + full), not the whole 6-arm
grid — transformer word-embeddings are known to underperform W2V on isolated-word similarity, so
this is a comparison point, not the headline. **DistilBERT's real role in the thesis is the
fusion classifier in Block I**, not a word-similarity backend.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

_tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')
_bert = AutoModel.from_pretrained('distilbert-base-uncased').eval()
_dev = 'cuda' if torch.cuda.is_available() else 'cpu'; _bert.to(_dev)

@torch.no_grad()
def _embed_batch(strings):
    enc = _tok(strings, padding=True, truncation=True, max_length=16, return_tensors='pt').to(_dev)
    out = _bert(**enc).last_hidden_state                       # B x T x 768
    mask = enc['attention_mask'].unsqueeze(-1)                 # mean-pool over real tokens
    v = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
    return v.cpu().numpy()

# cache one vector per distinct string (tags + every context term + adjective)
_need = sorted({s for t in lexicon for s in [t]} |
               {s for r in rows for s in r['context_nouns'] + r['adjectives']})
_BCACHE = f'{ROOT}/models_cache/distilbert_h5_subset.pkl'
if os.path.exists(_BCACHE):
    BVEC = pickle.load(open(_BCACHE, 'rb'))
else:
    BVEC = {}
    for i in range(0, len(_need), 128):
        chunk = _need[i:i + 128]
        for s, v in zip(chunk, _embed_batch([x.replace('_', ' ').replace('-', ' ') for x in chunk])):
            BVEC[s] = v.astype(np.float32)
    pickle.dump(BVEC, open(_BCACHE, 'wb'))
print(f'{len(BVEC)} DistilBERT vectors cached')

def bert_vec(s):
    if s in BVEC:
        return BVEC[s]
    sub = [BVEC[w] for w in parts(s) if w in BVEC]
    return np.mean(sub, axis=0) if sub else _embed_batch([s.replace('_', ' ').replace('-', ' ')])[0]

BTAG_M = np.vstack([bert_vec(t) for t in TAGS])
BTAG_M /= np.linalg.norm(BTAG_M, axis=1, keepdims=True) + 1e-9

def run_arm_bert(drop_abstract=False, add_adj=False, clustered=False):
    ranks = []
    for r in rows:
        ranked, sc = noun_to_tag(arm_terms(r, drop_abstract, add_adj),
                                 tag_m=BTAG_M, embed=bert_vec)
        if clustered:
            cl, flat = tag_to_tag_B(ranked, tag_m=BTAG_M)
            ranks.append(rank_of(r['gold_tag'], flat, clusters=cl))
        else:
            ranks.append(rank_of(r['gold_tag'], ranked, scores=sc))
    return metrics(ranks)

BARMS = {'0 baseline': dict(), '2 + abstract-noun removal': dict(drop_abstract=True),
         '5 full (2+adj)': dict(drop_abstract=True, add_adj=True)}
bres = {}
for title, clustered in [('DistilBERT  (Algorithm 2)', False),
                         ('DistilBERT-Clustered', True)]:
    print(f'{title:-^72}')
    header()
    for name, kw in BARMS.items():
        bres[f'{title} | {name}'] = run_arm_bert(clustered=clustered, **kw)
        show(name, bres[f'{title} | {name}'])
    print()
print('Yousaf 2023 BERT (DeBERTa): H@1 0.1038  H@5 0.3491  nDCG@20 0.2911')
json.dump(bres, open(f'{ROOT}/results_h/h5_distilbert.json', 'w'), indent=1)


In [ ]:
# ============================== H5 SUMMARY ==================================
w2v_plain = json.load(open(f'{ROOT}/results_h/h5_plain_w2v.json'))
w2v_clust = json.load(open(f'{ROOT}/results_h/h5_clustered_w2v.json'))

print('HEADLINE METRIC: Hits@3  (supervisor directive)\n')
print(f'{"":34}{"H@1":>8}{"H@3":>8}{"H@5":>8}{"nDCG@20":>10}')
print('-' * 68)
print(f'{"BARS TO CLEAR":<34}')
print(f'{"  Yousaf 2023 best (W2V Clust.)":<34}{0.2264:>8.4f}{"--":>8}{0.5283:>8.4f}{0.4442:>10.4f}')
print(f'{"  our reproduction (= H3)":<34}{0.2358:>8.4f}{0.4623:>8.4f}{0.5283:>8.4f}{0.4579:>10.4f}')
print(f'{"  frequency prior (test-peeking)":<34}{0.2830:>8.4f}{0.5189:>8.4f}{0.6604:>8.4f}{0.5624:>10.4f}')
print()
print(f'{"OUR METHOD (best per family)":<34}')
best = {
    'W2V + abstract-noun removal':      w2v_plain['2 + abstract-noun removal'],
    'W2V + abstr.rm + hint-boost (A4)':  w2v_plain['4 = 2 + 3'],
    'W2V-Clust + abstract-noun removal': w2v_clust['2 + abstract-noun removal'],
    'W2V-Clust full (2+3+adj) (A5)':     w2v_clust['5 full (2+3+adj)'],
}
for name, m in best.items():
    print(f'{"  " + name:<34}{m["Hits@1"]:>8.4f}{m["Hits@3"]:>8.4f}'
          f'{m["Hits@5"]:>8.4f}{m["nDCG@20"]:>10.4f}')

print(f"""
READ:
 * Best nDCG@20 = {max(m['nDCG@20'] for m in best.values()):.4f}  (W2V-Clust + abstract-noun removal)
   -> beats Yousaf published 0.4442 AND our reproduction 0.4579.  A real gain.
 * Best Hits@3  = {max(m['Hits@3'] for m in best.values()):.4f}
   -> beats our reproduced baseline 0.4623, but NOT the frequency prior 0.5189.
 * "+ all adjectives" dilutes on EVERY backend (W2V -0.003 nDCG, DistilBERT -0.11 nDCG).
   Confirmed: dumping raw adjectives into the similarity sum does not work.
 * "abstract-noun removal" (the authors' own stated future work) is the single
   biggest lever: +0.034 nDCG plain, +0.019 clustered.

CONCLUSION -> the adjective *layer* alone is not enough. The win needs SELECTION:
mine which adjective-noun patterns predict which tags (FP-Growth) and keep only
the predictive ones (Elastic Net). That is Block I, and this is its motivation.
""")

json.dump({'w2v_plain': w2v_plain, 'w2v_clustered': w2v_clust,
           'distilbert': json.load(open(f'{ROOT}/results_h/h5_distilbert.json')),
           'bars': {'yousaf_best_nDCG': 0.4442, 'reproduction_nDCG': 0.4579,
                    'freq_prior_hits3': 0.5189, 'reproduction_hits3': 0.4623}},
          open(f'{ROOT}/results_h/h5_summary.json', 'w'), indent=1)
print('wrote results_h/h5_summary.json')
